# LocateAnything-3B Live Webcam on Google Colab
This notebook installs the required dependencies and launches a live Gradio web app so you can stream your local webcam to the Colab GPU for fast real-time object grounding.

In [ ]:
!pip install torch torchvision transformers peft huggingface-hub opencv-python pillow gradio
!git clone https://github.com/NVlabs/Eagle.git
import sys
sys.path.append('/content/Eagle/Embodied')

In [ ]:
import gradio as gr
import cv2
import numpy as np
from PIL import Image
import torch
from locateanything_worker import LocateAnythingWorker

# 1. Load the model on the Colab GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading LocateAnything on {device}...")
worker = LocateAnythingWorker("nvidia/LocateAnything-3B", device=device)
print("Model loaded successfully!")

# Define the categories you want to find
categories = ["person", "phone", "coffee mug"]
colors = {
    "person": (0, 255, 0),
    "phone": (0, 0, 255),
    "coffee mug": (255, 0, 0),
}

def process_frame(frame):
    pil_image = Image.fromarray(frame)
    w, h = pil_image.size
    
    output_frame = frame.copy()
    
    for category in categories:
        try:
            result = worker.ground_multi(pil_image, category)
            boxes = LocateAnythingWorker.parse_boxes(result["answer"], w, h)
            
            color = colors.get(category, (255, 255, 0))
            
            for box in boxes:
                x1, y1, x2, y2 = int(box["x1"]), int(box["y1"]), int(box["x2"]), int(box["y2"])
                cv2.rectangle(output_frame, (x1, y1), (x2, y2), color, 4)
                cv2.putText(output_frame, category, (x1, max(y1 - 10, 0)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3)
        except Exception as e:
            pass
            
    return output_frame

# 2. Launch the Web Interface
demo = gr.Interface(
    fn=process_frame,
    inputs=gr.Image(sources=["webcam"], streaming=True),
    outputs="image",
    live=True,
    title="LocateAnything-3B Live Grounding (Colab GPU)"
)

demo.launch(share=True)